# Employee Attrition Simulation (JPSED)

労働者の属性から**翌年の離職**を予測する。母集団は**雇用者のみ**。目的変数は2種を構築して比較する:

- **実離職 `attrition_separation`**: 翌年調査で「仕事を辞めた・退職した」(Q57_1/Q58_1) と申告 → 実際の離職行動 (基準率~8%)。
- **転職・就職意向 `attrition_intention`**: 当年の Q106/Q105 が積極的意向 (コード1-2) → 意思 (基準率~19%)。

Colab / ローカルどちらでも動く。`DATA_MODE` でデータソースを切替。

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
# 'synthetic' : 同梱の合成データ（申請不要）
# 'gdrive'    : Google Drive 上の JPSED データ（Colab）
# 'local'     : ローカルの JPSED データ
DATA_MODE = 'synthetic'

GDRIVE_DATA_DIR = '/content/drive/MyDrive/ColabNotebooks/employee-attrition/data'
LOCAL_DATA_DIR  = '/Users/yoshikawadaisuke/My Drive/ColabNotebooks/employee-attrition/data'
REPO_URL        = 'https://github.com/dsk-yshkw/employee-attrition'

In [ ]:
# ── Install / clone repo (Colab) ───────────────────────────────────────────────
import subprocess, sys, os
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if not os.path.exists('employee-attrition'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('employee-attrition')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'], check=True)
    if DATA_MODE == 'gdrive':
        from google.colab import drive
        drive.mount('/content/drive')

# repo ルートを import パスに追加（ローカル実行で notebooks/ から開いた場合に対応）
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

In [ ]:
# ── Resolve data directory ─────────────────────────────────────────────────────
SYNTHETIC = (DATA_MODE == 'synthetic')
if SYNTHETIC:
    DATA_DIR = 'data/synthetic'
    # 合成の年次CSVが無ければ生成
    if not os.path.exists('data/synthetic/2022.csv'):
        subprocess.run([sys.executable, 'data/synthetic/generate_synthetic.py'], check=True)
elif DATA_MODE == 'gdrive':
    DATA_DIR = GDRIVE_DATA_DIR
else:
    DATA_DIR = LOCAL_DATA_DIR
print('DATA_MODE :', DATA_MODE)
print('DATA_DIR  :', DATA_DIR)

## 1. パネル構築とラベル（母集団=雇用者）

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.data.loader import DataLoader
from src.data.panel_builder import PanelBuilder
from src.features.assembler import FeatureAssembler
from src.config import TARGET_SEPARATION, TARGET_INTENTION

loader = DataLoader(DATA_DIR, synthetic=SYNTHETIC)
pb = PanelBuilder(loader)
fa = FeatureAssembler()

panel = pb.add_targets(pb.build())
emp = pb.employee_mask(panel)
sub = panel[emp]
print('person-year rows:', len(panel), '| employees:', int(emp.sum()),
      '| persons:', panel['pkey'].nunique())
print('陽性率  separation=%.3f  intention=%.3f' %
      (sub[TARGET_SEPARATION].dropna().mean(), sub[TARGET_INTENTION].dropna().mean()))
sub.groupby('year')[[TARGET_SEPARATION, TARGET_INTENTION]].mean().round(3)

## 1b. 名目 vs 実質年収の推移（インフレによる実質賃金の目減り）

JPSEDのwave Yは前年(Y-1)の年収を尋ねるため、CPIはY-1年で実質化。2022–2024で名目は上昇するが実質は下落する。

In [ ]:
from src.data.macro import MacroFeatureBuilder
X_macro = MacroFeatureBuilder().build(sub)
tmp = sub.assign(nominal=X_macro['real_annual_income']*0 + sub['annual_income'].astype(float),
                 real=X_macro['real_annual_income'], cpi=X_macro['cpi'])
g = tmp.groupby('year')[['nominal','real','cpi']].mean()
print(g.round(1).to_string())
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(g.index, g['nominal'], 'o-', label='Nominal income (man-yen)')
ax.plot(g.index, g['real'],    's-', label='Real income (2020 yen)')
ax.set_xlabel('survey year'); ax.set_ylabel('mean annual income'); ax.legend(); ax.set_title('Nominal vs Real income')
plt.tight_layout(); plt.show()

## 2. モデル学習・評価（時系列split, 2ターゲット × 複数モデル）

In [ ]:
from src.pipeline import run_experiment, _xgboost_available
models = ['logistic', 'histgbm'] + (['xgboost'] if _xgboost_available() else [])
plans  = [(TARGET_SEPARATION, 2024, 'Separation'), (TARGET_INTENTION, 2025, 'Intention')]

rows, results = [], {}
for target, test_year, name in plans:
    for m in models:
        r = run_experiment(pb, fa, target, test_year, model_type=m)
        results[(target, m)] = r
        rows.append(dict(target=name, model=m, test_year=test_year,
                         n_train=r.n_train, n_test=r.n_test,
                         pos_rate=round(r.test_pos_rate, 3),
                         AUC=round(r.metrics['auc_roc'], 3),
                         PR_AUC=round(r.metrics['auc_pr'], 3)))
summary = pd.DataFrame(rows)
best = {t: max(models, key=lambda m: results[(t, m)].metrics['auc_roc']) for t, _, _ in plans}
summary

## 3. ROC / PR 曲線（各ターゲットのベストモデル）

In [ ]:
from src.models.evaluator import ModelEvaluator
ev = ModelEvaluator()
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for i, (target, test_year, name) in enumerate(plans):
    fr = pb.build_modeling_frame(target)
    te = fr[fr['year'] == test_year]
    yp = results[(target, best[target])].model.predict_proba(fa.build(te))
    ev.plot_roc(te[target], yp, ax=axes[i, 0]); axes[i, 0].set_title(f'{name} ROC ({best[target]})')
    ev.plot_pr(te[target], yp, ax=axes[i, 1]);  axes[i, 1].set_title(f'{name} PR ({best[target]})')
plt.tight_layout(); plt.show()

## 4. 特徴量重要度（Permutation importance）

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (target, test_year, name) in zip(axes, plans):
    imp = results[(target, best[target])].importance.sort_values().tail(12)
    imp.plot(kind='barh', ax=ax, color='#55A868')
    ax.set_title(f'{name}: {best[target]} 重要度 top12'); ax.set_xlabel('AUC drop')
plt.tight_layout(); plt.show()

## 6. 遷移サブモデル（多年シミュレーションの構成要素）

1年遷移を回すため、離職に加えて再就業・賃金の遷移をモデル化する:

| サブモデル | 対象 | 目的変数 |
|---|---|---|
| 離職 | 雇用者 | 翌年に辞めたか |
| 再就業 | 離職者 | 翌12月に就業か |
| 継続者賃金 | 継続者 | 翌年の実質年収 |
| 転職後賃金 | 離職→再就業 | 翌年の実質年収 |

In [ ]:
from src.data.transitions import TransitionFrameBuilder
from src.models.transitions import TransitionModels

tf, tfeats = TransitionFrameBuilder(pb).build()
print('transition rows:', len(tf), '| features:', len(tfeats))
print('base: separated=%.3f  employed_next=%.3f  reemploy|sep=%.3f' % (
    tf['separated'].mean(), tf['employed_next'].mean(),
    tf.loc[tf.separated==1,'employed_next'].mean()))

split = tf['year'].max() - 1           # hold out the latest transition year
trm = TransitionModels(tfeats).fit(tf[tf['year']<split])
val = trm.validate(tf[tf['year']==split])
import pandas as pd
pd.Series(val).round(3).to_frame(f'test={split}')

## 7. 多年マイクロシミュレーション（バックテスト＋インフレ・シナリオ）

遷移モデルを使い、基準年の雇用者コホートを前進反復。離職率の多年推移を生成し、①実測とのバックテスト、②高インフレ・シナリオでの反応を見る。

In [ ]:
from src.models.microsim import MicroSimulator, initial_state, inflation_scenario
from src.data.macro import MacroData

yrs = sorted(tf['year'].unique())
init_year = max(yrs[0]+1, yrs[-1]-3)        # leave >=3 years to simulate if available
n_steps   = yrs[-1] - init_year + 1
tm_sim = TransitionModels(tfeats).fit(tf[tf['year'] < init_year])
print('train transitions < %d ; simulate %d..%d' % (init_year, init_year, yrs[-1]))

actual = sub.groupby('year')[TARGET_SEPARATION].mean()
sim = MicroSimulator(tm_sim, tfeats, MacroData())
base = sim.simulate(initial_state(pb, fa, init_year), init_year, n_steps, seed=1, wage_noise_sd=30.0)
base['actual_attrition'] = base['year'].map(actual)
print(base[['year','attrition_rate','actual_attrition','mean_nominal_income','mean_real_income','pop']].round(3).to_string(index=False))

In [ ]:
# Backtest plot + inflation scenario
hi = inflation_scenario(MacroData(), 3.0, list(range(init_year, yrs[-1]+1)))
scn = sim.simulate(initial_state(pb, fa, init_year), init_year, n_steps, seed=1, cpi_override=hi, wage_noise_sd=30.0)

fig, ax = plt.subplots(1, 2, figsize=(13,4.5))
ax[0].plot(base['year'], base['attrition_rate'], 'o-', label='simulated')
ax[0].plot(base['year'], base['actual_attrition'], 's--', label='actual')
ax[0].set_title('Backtest: attrition rate'); ax[0].set_xlabel('year'); ax[0].legend(); ax[0].set_ylim(0, None)
ax[1].plot(base['year'], base['mean_real_income'], 'o-', label='baseline CPI')
ax[1].plot(scn['year'], scn['mean_real_income'], '^-', label='+3pp inflation')
ax[1].set_title('Scenario: mean real income'); ax[1].set_xlabel('year'); ax[1].legend()
plt.tight_layout(); plt.show()

comp = base[['year','attrition_rate']].merge(scn[['year','attrition_rate']], on='year', suffixes=('_base','_hi'))
print('attrition under +3pp inflation vs baseline:'); print(comp.round(3).to_string(index=False))

## 8. 系列モデル（GRU / Transformer）vs 木系

雇用履歴（系列）が単年状態より予測を改善するかを検証。MLP(単年)・GRU・Transformerを同一の標準化特徴で比較し、木系を参照。※実データでは学習に数分。`RUN_SEQUENCE=False` でスキップ可。

In [ ]:
RUN_SEQUENCE = True   # set False to skip (slow on real data)
if RUN_SEQUENCE:
    from src.models.sequence import run_comparison
    TESTY = int(tf['year'].max())          # last transition year
    aucs, info = run_comparison(tf, tfeats, TESTY, max_len=8, epochs=6, seed=0)
    from src.models.attrition import AttritionModel
    from src.models.evaluator import ModelEvaluator
    tr, teX = tf[tf['year']<TESTY], tf[tf['year']==TESTY]
    ev = ModelEvaluator()
    for mt in ['histgbm','xgboost']:
        try:
            m = AttritionModel(model_type=mt).fit(tr[tfeats], tr['separated'])
            aucs['tree_'+mt] = ev.evaluate(teX['separated'], m.predict_proba(teX[tfeats]))['auc_roc']
        except Exception as e:
            print('skip',mt,e)
    print(f'next-year separation AUC (test={TESTY}), n_test={info["n_test"]}')
    import pandas as pd
    display(pd.Series(aucs).round(4).sort_values(ascending=False).to_frame('AUC'))
else:
    print('sequence comparison skipped')

## 9. 労働供給弾力性（RUM離散選択解釈）＋ 標本ウェイト

離職ロジットをランダム効用の継続選択とみなし、実質賃金+10%に対する**就業継続の賃金弾力性**（外延的労働供給）を算出。非正規ほど賃金に敏感という異質性が出る。クロスセクションウェイトで頑健性も確認。

In [ ]:
from src.models.attrition import AttritionModel
from src.models.elasticity import retention_elasticity, elasticity_by_group
import pandas as pd, numpy as np

TESTY = int(tf['year'].max())
logit = AttritionModel(model_type='logistic', class_weight=None).fit(
    tf[tf['year']<TESTY][tfeats], tf[tf['year']<TESTY]['separated'])
psep = lambda X: logit.predict_proba(X)
te = tf[tf['year']==TESTY].reset_index(drop=True); X = te[tfeats]

r = retention_elasticity(psep, X, pct=0.10)
print('Retention wage elasticity (+10% real wage): %.4f  (semi-elasticity %.4f pp/log-wage)'
      % (r['elasticity'], r['semi_elasticity']))
g = elasticity_by_group(psep, X, te['contract_type'].fillna(-1).astype(int), 0.10)
g['contract'] = g['group'].map({1:'正規',2:'パート',3:'派遣',4:'契約',5:'嘱託',6:'その他'})
display(g[['contract','n','retention_base','elasticity']].round(4))

# weighted vs unweighted attrition rate (cross-section weight)
w = pd.to_numeric(sub['cross_section_weight'], errors='coerce')
d = sub.dropna(subset=[TARGET_SEPARATION])
wr = np.average(d[TARGET_SEPARATION], weights=pd.to_numeric(d['cross_section_weight'],errors='coerce').fillna(0).clip(lower=0)+1e-9)
print('attrition rate  unweighted=%.4f  weighted=%.4f' % (d[TARGET_SEPARATION].mean(), wr))

## 10. 特徴量分析：SHAP（TreeSHAP）

離職モデル(xgboost, native categorical)の SHAP で、大域重要度＋方向＋dependence を見る。**実質賃金→離職**の dependence は弾力性(§9)の視覚的裏づけ。相関特徴(勤続・年齢・年収)は合算的に解釈。

In [ ]:
from src.models.interpret import fit_xgb_for_shap, shap_analysis, dependence
TESTY = int(tf['year'].max())
xgb_shap = fit_xgb_for_shap(tf[tf['year']<TESTY][tfeats], tf[tf['year']<TESTY]['separated'])
sv, imp, expl, Xs = shap_analysis(xgb_shap, tf[tf['year']==TESTY][tfeats], max_samples=4000, seed=0)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
imp.head(12)[::-1].plot(kind='barh', ax=ax[0], color='#4C72B0')
ax[0].set_title('SHAP global importance (top 12)'); ax[0].set_xlabel('mean |SHAP|')
for a, feat in zip(ax[1:], ['real_annual_income', 'tenure_years']):
    d = dependence(sv, Xs, feat)
    a.scatter(d['value'], d['shap'], s=5, alpha=0.2, color='#55A868')
    a.axhline(0, color='k', lw=0.6); a.set_title(f'SHAP dependence: {feat}')
    a.set_xlabel(feat); a.set_ylabel('SHAP (log-odds of separation)')
    if feat == 'real_annual_income': a.set_xlim(0, 1200)
plt.tight_layout(); plt.show()
display(imp.round(3).to_frame('mean|SHAP|').head(12))

In [ ]:
# Panel-attrition weighting robustness (real data only; synthetic has no weight files)
import numpy as np
if not USE_SYNTHETIC:
    from src.data.weights import AttritionWeights
    aw = AttritionWeights(DATA_DIR).attach(tf)
    twf = tf.assign(aw=aw.values)
    cov = (twf['aw'] > 0).mean()
    d = twf[twf['aw'] > 0]
    byyr = pd.DataFrame({
        'unweighted': tf.groupby('year')['separated'].mean(),
        'attr_weighted': d.groupby('year').apply(lambda g: np.average(g['separated'], weights=g['aw'])),
    })
    print('attrition-weight coverage: %.1f%%' % (100*cov))
    display(byyr.round(4))
else:
    print('attrition weights require the licensed weight files (real data)')

## 5. 考察

- **実離職**の主要予測子は勤続年数(tenure)・年齢・年収・雇用形態。
- **意向**は雇用形態・年齢の寄与が相対的に大きい。
- **年収増加率**の寄与は小さい。定期昇給と昇進/転職/労働時間変化が混在した粗い指標（JPSEDに基本給と定期昇給の内訳が無い）。
- 実離職は稀事象(~8%)でPR-AUCが低め。実務では閾値調整・class weight・コスト考慮が重要。

> 実データは SSJDA 利用許諾のため非公開。`DATA_MODE` で合成/実データを切替可能。